# 09. 성견 TF-IDF 우위 가설 정량 검증

**교수님 Q2**: *"성견 구간에서 TF-IDF가 BERT보다 우수한 이유가 무엇인가?"*

**가설**: 성견 보호자 질문은 자견/노령견보다 구어체 표현이 적고 의학 용어(명사) 비율이 높아,  
키워드 매칭에 강한 TF-IDF가 상대적으로 유리하다.

**검증 지표**
1. 구어체 정규화 적용률 (생애주기별)
2. COLLOQUIAL_MAP 키워드 등장 빈도 (생애주기별)
3. 질문 텍스트 길이 분포 (생애주기별)
4. 원본/정규화 텍스트 변화량 (생애주기별)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from scipy import stats

# 한글 폰트 설정
try:
    plt.rcParams['font.family'] = 'AppleGothic'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data' / 'processed'

print('설정 완료')

In [ ]:
# 03_preprocessing.ipynb 와 동일한 COLLOQUIAL_MAP
COLLOQUIAL_MAP = {
    '강아지': '반려견',
    '댕댕이': '반려견',
    '멍멍이': '반려견',
    '토해요': '구토',
    '토했어요': '구토',
    '안 먹어요': '식욕부진',
    '안먹어요': '식욕부진',
    '밥 안 먹어요': '식욕부진',
    '눈물': '눈 분비물',
    '절뚝': '파행',
    '절뚝거려요': '파행',
    '긁어요': '소양감',
    '긁는다': '소양감',
    '숨을 헐떡거려요': '호흡곤란',
    '헐떡거려요': '호흡곤란',
    '헐떡여요': '호흡곤란',
}

corpus = pd.read_csv(DATA / 'corpus_preprocessed.csv')
print(f'전체 코퍼스: {len(corpus):,}건')
print(corpus['lifeCycle'].value_counts())

## 1. 구어체 정규화 적용률 (생애주기별)

In [ ]:
# 구어체 정규화 여부: input_clean ≠ input_normalized 이면 구어체 존재
corpus['has_colloquial'] = corpus['input_clean'] != corpus['input_normalized']

# 생애주기별 구어체 비율
colloquial_rate = corpus.groupby('lifeCycle')['has_colloquial'].agg(['mean', 'sum', 'count'])
colloquial_rate.columns = ['구어체 비율', '구어체 건수', '전체 건수']
colloquial_rate['구어체 비율(%)'] = (colloquial_rate['구어체 비율'] * 100).round(2)
print('=== 생애주기별 구어체 정규화 적용률 ===')
print(colloquial_rate[['구어체 비율(%)', '구어체 건수', '전체 건수']])

## 2. COLLOQUIAL_MAP 키워드별 생애주기 빈도

In [ ]:
# 각 COLLOQUIAL_MAP 키워드가 생애주기별로 얼마나 자주 등장하는지
rows = []
for key in COLLOQUIAL_MAP:
    for lc in ['자견', '성견', '노령견']:
        sub = corpus[corpus['lifeCycle'] == lc]
        count = sub['input_clean'].str.contains(key, na=False).sum()
        rate  = count / len(sub) * 100
        rows.append({'키워드': key, '생애주기': lc, '등장 건수': count, '등장률(%)': round(rate, 2)})

kw_df = pd.DataFrame(rows)
pivot = kw_df.pivot(index='키워드', columns='생애주기', values='등장률(%)')
pivot['성견-자견 차이'] = pivot['성견'] - pivot['자견']
pivot = pivot.sort_values('성견-자견 차이')

print('=== 키워드별 등장률(%) — 성견 vs 자견 ===')
print(pivot.round(3))

## 3. 시각화 — 구어체 비율 & 키워드 히트맵

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 구어체 비율 막대그래프
lc_order = ['자견', '성견', '노령견']
rates = [colloquial_rate.loc[lc, '구어체 비율(%)'] for lc in lc_order]
colors = ['#4C9BE8', '#E85C5C', '#5CBF7A']
bars = axes[0].bar(lc_order, rates, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title('생애주기별 구어체 표현 포함 비율', fontsize=13, fontweight='bold')
axes[0].set_ylabel('구어체 포함 질문 비율 (%)')
axes[0].set_ylim(0, max(rates) * 1.3)
axes[0].axhline(y=sum(rates)/len(rates), color='gray', linestyle='--', alpha=0.6, label='전체 평균')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

# 오른쪽: 키워드 히트맵 (상위 10개)
top_keys = kw_df.groupby('키워드')['등장 건수'].sum().nlargest(10).index
heat_data = pivot.loc[pivot.index.isin(top_keys), ['자견', '성견', '노령견']]
im = axes[1].imshow(heat_data.values, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(['자견', '성견', '노령견'])
axes[1].set_yticks(range(len(heat_data)))
axes[1].set_yticklabels(heat_data.index)
axes[1].set_title('COLLOQUIAL_MAP 키워드 등장률 히트맵 (%)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='등장률 (%)')
for i in range(len(heat_data)):
    for j in range(3):
        val = heat_data.values[i, j]
        axes[1].text(j, i, f'{val:.1f}', ha='center', va='center',
                     color='white' if val > heat_data.values.max()*0.6 else 'black', fontsize=9)

plt.tight_layout()
plt.savefig(DATA / 'eda_figures' / 'q2_colloquial_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장 완료')

## 4. 질문 텍스트 길이 분포 (생애주기별)

In [ ]:
corpus['input_len']  = corpus['input_clean'].str.len()
corpus['token_count'] = corpus['input_tokens'].str.split().str.len()

length_stat = corpus.groupby('lifeCycle').agg(
    평균_글자수=('input_len', 'mean'),
    중앙값_글자수=('input_len', 'median'),
    평균_토큰수=('token_count', 'mean'),
    중앙값_토큰수=('token_count', 'median'),
).round(1)
print('=== 생애주기별 질문 길이 ===')
print(length_stat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lc, color in zip(['자견', '성견', '노령견'], colors):
    sub = corpus[corpus['lifeCycle'] == lc]
    axes[0].hist(sub['input_len'].clip(upper=500), bins=40, alpha=0.5, label=lc, color=color)
    axes[1].hist(sub['token_count'].clip(upper=60), bins=30, alpha=0.5, label=lc, color=color)

axes[0].set_title('질문 글자수 분포', fontsize=13, fontweight='bold')
axes[0].set_xlabel('글자 수')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].set_title('형태소 토큰 수 분포', fontsize=13, fontweight='bold')
axes[1].set_xlabel('토큰 수')
axes[1].legend()
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(DATA / 'eda_figures' / 'q2_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 구어체 변화량 정량 분석 (Kruskal-Wallis 검정)

In [ ]:
# 구어체 변화량: 정규화 전후 글자 수 차이
corpus['norm_delta'] = corpus['input_clean'].str.len() - corpus['input_normalized'].str.len()

groups = [corpus[corpus['lifeCycle'] == lc]['norm_delta'].values for lc in ['자견', '성견', '노령견']]
stat, p = stats.kruskal(*groups)

print('=== 생애주기별 구어체 변화량 (정규화 전후 글자 수 차이) ===')
for lc, g in zip(['자견', '성견', '노령견'], groups):
    print(f'  {lc}: 평균 {g.mean():.2f}자, 중앙값 {np.median(g):.1f}자, 변화 있음 {(g>0).mean():.1%}')

print(f'\nKruskal-Wallis: H={stat:.3f}, p={p:.4f}')
if p < 0.05:
    print('✅ p < 0.05: 생애주기별 구어체 사용량 차이가 통계적으로 유의미')
else:
    print('❌ p ≥ 0.05: 생애주기별 유의미한 차이 없음')

## 6. 결론 — Q2 답변 요약

In [ ]:
print('=' * 60)
print('교수님 Q2 답변: 성견 TF-IDF 우위 가설 검증 결과')
print('=' * 60)

for lc in ['자견', '성견', '노령견']:
    sub = corpus[corpus['lifeCycle'] == lc]
    rate = sub['has_colloquial'].mean() * 100
    delta = sub['norm_delta'].mean()
    print(f'  {lc}: 구어체 포함 {rate:.1f}%, 평균 변화량 {delta:.2f}자')

print()
자견_rate = corpus[corpus['lifeCycle']=='자견']['has_colloquial'].mean()*100
성견_rate = corpus[corpus['lifeCycle']=='성견']['has_colloquial'].mean()*100
노령견_rate = corpus[corpus['lifeCycle']=='노령견']['has_colloquial'].mean()*100

if 성견_rate < 자견_rate:
    print(f'→ 성견 질문의 구어체 비율({성견_rate:.1f}%)이 자견({자견_rate:.1f}%)보다 낮음')
    print('→ 가설 지지: 성견 보호자가 보다 formal한 의학 표현 사용')
    print('→ TF-IDF는 exact keyword 매칭에 강해, 의학 용어가 많은 성견 질문에서 상대적 우위')
else:
    print(f'→ 생애주기별 구어체 비율 차이 미미 (자견 {자견_rate:.1f}%, 성견 {성견_rate:.1f}%)')
    print('→ 구어체 빈도 외 다른 요인 추가 분석 필요 (질문 길이, 의학 명사 밀도 등)')